# Markowitz Portfolio Optimization
## Modern Portfolio Theory — From First Principles

> *"It is not enough to simply pick the securities with the highest expected returns; one must consider how they interact."* — Harry Markowitz, 1952

---

This notebook implements the complete Markowitz mean-variance framework, walking from raw price data through to optimal portfolio allocations. Financial intuition is explained at every step for both technical and non-technical readers.

### What we build
1. **Data** — 3 years of adjusted-close prices for 8 diversified assets via `yfinance`
2. **Returns** — daily arithmetic returns, annualised μ and Σ
3. **Monte Carlo** — 10,000 random long-only portfolios to visualise the feasible set
4. **Optimisation** — scipy SLSQP to find the Max Sharpe and Min Variance portfolios
5. **Efficient Frontier** — the set of every undominated portfolio
6. **Visualisations** — three publication-quality charts saved to `./plots/`

### Asset Universe
| Ticker | Name | Role |
|--------|------|------|
| AAPL | Apple | Growth — Technology |
| MSFT | Microsoft | Growth — Technology / Cloud |
| AMZN | Amazon | Growth — E-commerce / Cloud |
| GOOGL | Alphabet | Growth — Advertising / AI |
| JNJ | Johnson & Johnson | Defensive — Healthcare |
| JPM | JPMorgan Chase | Cyclical — Financials |
| GLD | SPDR Gold Trust | Safe-haven — Commodity |
| ^GSPC | S&P 500 | Broad market benchmark |

---
## 1. Setup & Imports

All dependencies are listed in `requirements.txt`.  Run the cell below to install them if needed, then import.

In [ ]:
# Uncomment to install:
# !pip install -r requirements.txt

In [ ]:
import os
from datetime import datetime, timedelta
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from IPython.display import Image, display
from scipy.optimize import minimize

# Global constants
TICKERS = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'JNJ', 'JPM', 'GLD', '^GSPC']
RISK_FREE_RATE = 0.05   # 5% annualised — approximate US 10-yr Treasury yield
TRADING_DAYS = 252      # standard annualisation factor for equity markets
N_SIMULATIONS = 10_000  # Monte Carlo sample count
YEARS = 3               # historical look-back window
PLOTS_DIR = 'plots'

# Consistent figure style throughout the notebook
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'grid.alpha': 0.4,
})
os.makedirs(PLOTS_DIR, exist_ok=True)
print('Setup complete.')

---
## 2. Data Collection

We download 3 years of **adjusted-close prices** from Yahoo Finance using `yfinance`.

*Why adjusted close?*  yfinance's `auto_adjust=True` applies split and dividend adjustments, giving us **total-return** prices.  Using unadjusted prices would understate returns for dividend-paying assets (e.g. JNJ, JPM) and misrepresent risk metrics.

*Missing data:*  Different assets may not trade on the same days (e.g. GLD trades on days the equity market is closed, and vice versa).  We forward-fill any gaps then drop remaining leading NaNs, keeping the longest clean common history.

In [ ]:
def download_data(tickers, years=YEARS):
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))

    raw = yf.download(
        tickers,
        start=start.strftime('%Y-%m-%d'),
        end=end.strftime('%Y-%m-%d'),
        auto_adjust=True,
        progress=False,
        threads=True,
    )
    # yfinance 0.2+ returns MultiIndex columns: level-0 = metric, level-1 = ticker
    prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
    return prices.ffill().dropna()

In [ ]:
prices = download_data(TICKERS)
# Clean display: strip '^' from index symbol names
prices.columns = [c.replace('^', '') for c in prices.columns]
clean_tickers = list(prices.columns)

print(f'Downloaded {len(prices):,} trading days for {prices.shape[1]} assets')
print(f'Date range : {prices.index[0].date()} → {prices.index[-1].date()}')
print()
display(prices.tail())

---
## 3. Computing Returns & Annualised Statistics

### Why arithmetic returns?
Portfolio return is a **weighted sum** of asset returns:
$$R_p = \sum_i w_i R_i = \mathbf{w}^\top \mathbf{R}$$
This linearity holds **exactly** for arithmetic (simple) returns, and only approximately for log-returns — so arithmetic returns are the correct choice for portfolio construction.

### Annualisation
Under the assumption of i.i.d. daily returns:
$$\mu_{annual} = \mu_{daily} \times 252$$
$$\Sigma_{annual} = \Sigma_{daily} \times 252$$
(Variance scales linearly with time; volatility σ therefore scales as √252.)

In [ ]:
def compute_returns(prices):
    return prices.pct_change().dropna()

def annualise_stats(returns):
    mu = returns.mean() * TRADING_DAYS
    sigma = returns.cov() * TRADING_DAYS
    return mu, sigma

returns = compute_returns(prices)
mu, sigma = annualise_stats(returns)

print('=== Annualised Expected Returns ===')
for ticker, ret in mu.items():
    print(f'  {ticker:<6}  {ret:>8.2%}')

print()
print('=== Annualised Covariance Matrix (first 4 assets) ===')
display(sigma.iloc[:4, :4].style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None))

---
## 4. Monte Carlo Simulation

Before optimising analytically, we sample 10,000 random portfolios to **visualise the feasible set** — the cloud of all achievable risk/return combinations.

Each weight vector is drawn from the **uniform Dirichlet distribution**: generate n uniform random numbers and divide by their sum.  This samples uniformly from the n-simplex (all weights ≥ 0, sum = 1) — the long-only constraint.

### Sharpe Ratio
$$S = \frac{R_p - R_f}{\sigma_p}$$

Measures excess return earned per unit of total risk.  The higher the Sharpe ratio, the more efficiently risk is being rewarded.  We use $R_f = 5\%$ throughout.

In [ ]:
def portfolio_performance(weights, mu, sigma):
    w = np.asarray(weights, dtype=float)
    ret = float(w @ mu)
    vol = float(np.sqrt(w @ sigma.values @ w))
    sharpe = (ret - RISK_FREE_RATE) / vol
    return ret, vol, sharpe

def monte_carlo_simulation(mu, sigma, n=N_SIMULATIONS, seed=42):
    np.random.seed(seed)
    n_assets = len(mu)
    sim_rets = np.empty(n)
    sim_vols = np.empty(n)
    sim_sharpes = np.empty(n)
    sim_weights = np.empty((n, n_assets))

    for i in range(n):
        w = np.random.random(n_assets)
        w /= w.sum()   # normalise → uniform Dirichlet sample
        sim_weights[i] = w
        sim_rets[i], sim_vols[i], sim_sharpes[i] = portfolio_performance(w, mu, sigma)

    return sim_rets, sim_vols, sim_sharpes, sim_weights

print('Running 10,000 Monte Carlo portfolio simulations ...')
sim_rets, sim_vols, sim_sharpes, sim_weights = monte_carlo_simulation(mu, sigma)
print(f'Done.  Sharpe range: [{sim_sharpes.min():.2f}, {sim_sharpes.max():.2f}]')
print(f'Return range : [{sim_rets.min():.2%}, {sim_rets.max():.2%}]')
print(f'Vol range    : [{sim_vols.min():.2%}, {sim_vols.max():.2%}]')

---
## 5. Portfolio Optimisation with scipy

We use `scipy.optimize.minimize` with `method='SLSQP'` (Sequential Least Squares Programming) — a gradient-based solver well-suited to quadratic objective functions with linear constraints.

### Constraints & Bounds
- **Equality**: $\sum_i w_i = 1$ (fully invested)
- **Bounds**: $w_i \in [0, 1]$ (long-only, no leverage, no short-selling)

### Automatic Retry
SLSQP can stall near the corners of the simplex where gradients are near-zero.  Our `_run_optimiser` helper retries with tighter bounds $[0.01, 0.50]$ if the first pass does not converge.

### Two Optimal Portfolios

| Portfolio | Objective | Description |
|-----------|-----------|-------------|
| **Max Sharpe** | Maximise $(R_p - R_f)/\sigma_p$ | Tangency portfolio — highest risk-adjusted return |
| **Min Variance** | Minimise $\sigma_p$ | Global Minimum Variance — most conservative |


In [ ]:
def _neg_sharpe(weights, mu, sigma):
    _, _, sharpe = portfolio_performance(weights, mu, sigma)
    return -sharpe

def _portfolio_vol(weights, mu, sigma):
    _, vol, _ = portfolio_performance(weights, mu, sigma)
    return vol

def _run_optimiser(objective, mu, sigma, x0=None, extra_constraints=None):
    n = len(mu)
    x0 = np.ones(n) / n if x0 is None else x0
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
    if extra_constraints:
        constraints.extend(extra_constraints)
    # Attempt 1: standard bounds; Attempt 2: tighter retry
    for bounds, maxiter in [
        (tuple((0.0, 1.0) for _ in range(n)), 1_000),
        (tuple((0.01, 0.50) for _ in range(n)), 2_000),
    ]:
        res = minimize(objective, x0, args=(mu, sigma), method='SLSQP',
                       bounds=bounds, constraints=constraints,
                       options={'ftol': 1e-9, 'maxiter': maxiter})
        if res.success:
            return res
    raise RuntimeError(f'Optimisation failed: {res.message}')

def max_sharpe_portfolio(mu, sigma):
    res = _run_optimiser(_neg_sharpe, mu, sigma)
    return res.x, *portfolio_performance(res.x, mu, sigma)

def min_variance_portfolio(mu, sigma):
    res = _run_optimiser(_portfolio_vol, mu, sigma)
    return res.x, *portfolio_performance(res.x, mu, sigma)

print('Optimising Max Sharpe portfolio ...')
ms_w, ms_ret, ms_vol, ms_sharpe = max_sharpe_portfolio(mu, sigma)
print(f'  Return     : {ms_ret:.2%}')
print(f'  Volatility : {ms_vol:.2%}')
print(f'  Sharpe     : {ms_sharpe:.2f}')
print()
print('Optimising Min Variance portfolio ...')
mv_w, mv_ret, mv_vol, mv_sharpe = min_variance_portfolio(mu, sigma)
print(f'  Return     : {mv_ret:.2%}')
print(f'  Volatility : {mv_vol:.2%}')
print(f'  Sharpe     : {mv_sharpe:.2f}')

---
## 6. The Efficient Frontier

The **Efficient Frontier** is the set of all portfolios that cannot be improved: no combination of these assets delivers higher expected return for the same risk, or lower risk for the same expected return.

We trace it by solving 200 minimum-variance problems, each with an additional constraint fixing the target return.  The range spans from the **Global Minimum Variance** return (the frontier's leftmost achievable point) up to the highest single-asset expected return.

Portfolios **below** the frontier use more risk than necessary for their return — they are *inefficient* and rational investors should avoid them.

In [ ]:
def efficient_frontier(mu, sigma, n_points=200):
    _, gmv_ret, _, _ = min_variance_portfolio(mu, sigma)
    max_ret = float(mu.max())
    target_rets = np.linspace(gmv_ret, max_ret, n_points)
    eff_rets, eff_vols = [], []
    for target in target_rets:
        ret_con = [{'type': 'eq',
                    'fun': lambda w, t=target: portfolio_performance(w, mu, sigma)[0] - t}]
        try:
            res = _run_optimiser(_portfolio_vol, mu, sigma, extra_constraints=ret_con)
            r, v, _ = portfolio_performance(res.x, mu, sigma)
            eff_rets.append(r)
            eff_vols.append(v)
        except RuntimeError:
            pass  # skip infeasible target returns
    return np.array(eff_rets), np.array(eff_vols)

print('Tracing efficient frontier (200 points) ...')
eff_rets, eff_vols = efficient_frontier(mu, sigma)
print(f'Computed {len(eff_rets)} frontier points.')

---
## 7. Visualisations

Three charts summarise the full analysis:

1. **Efficient Frontier Plot** — the 10,000 Monte Carlo portfolios as a scatter plot coloured by Sharpe ratio, with the efficient frontier overlaid and both optimal portfolios starred.

2. **Portfolio Weights Bar Chart** — side-by-side comparison of Max Sharpe and Min Variance allocations, making it immediately clear how the two objectives lead to different concentration patterns.

3. **Correlation Heatmap** — pairwise correlations that drive diversification benefits.  Low (or negative) correlations are the engine of MPT.

In [ ]:
def plot_efficient_frontier(sim_rets, sim_vols, sim_sharpes,
                             frontier_rets, frontier_vols,
                             ms_ret, ms_vol, mv_ret, mv_vol, plots_dir=PLOTS_DIR):
    fig, ax = plt.subplots(figsize=(12, 7))
    sc = ax.scatter(sim_vols, sim_rets, c=sim_sharpes, cmap='viridis',
                    alpha=0.45, s=8, label='10,000 random portfolios')
    plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
    ax.plot(frontier_vols, frontier_rets, color='crimson', lw=2.5, label='Efficient Frontier')
    ax.scatter(ms_vol, ms_ret, marker='*', color='gold', s=500, zorder=6,
               edgecolors='black', linewidths=0.5, label='Max Sharpe ★')
    ax.annotate('Max Sharpe', xy=(ms_vol, ms_ret),
                xytext=(10, 8), textcoords='offset points',
                fontsize=9, color='darkorange', fontweight='bold')
    ax.scatter(mv_vol, mv_ret, marker='*', color='deepskyblue', s=500, zorder=6,
               edgecolors='black', linewidths=0.5, label='Min Variance ★')
    ax.annotate('Min Variance', xy=(mv_vol, mv_ret),
                xytext=(10, -14), textcoords='offset points',
                fontsize=9, color='steelblue', fontweight='bold')
    ax.set_xlabel('Annual Volatility (Risk)', fontsize=12)
    ax.set_ylabel('Annual Expected Return', fontsize=12)
    ax.set_title('Markowitz Efficient Frontier\n10,000 Monte Carlo Portfolios — Coloured by Sharpe Ratio',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    path = os.path.join(plots_dir, 'efficient_frontier.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path

p1 = plot_efficient_frontier(sim_rets, sim_vols, sim_sharpes,
                              eff_rets, eff_vols, ms_ret, ms_vol, mv_ret, mv_vol)
display(Image(filename=p1))

In [ ]:
def plot_portfolio_weights(ms_weights, mv_weights, tickers, plots_dir=PLOTS_DIR):
    x = np.arange(len(tickers))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, ms_weights * 100, width, label='Max Sharpe',
           color='gold', edgecolor='black', linewidth=0.6)
    ax.bar(x + width/2, mv_weights * 100, width, label='Min Variance',
           color='steelblue', edgecolor='black', linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(tickers, rotation=45, ha='right', fontsize=11)
    ax.set_ylabel('Portfolio Allocation (%)', fontsize=12)
    ax.set_title('Optimal Portfolio Weights: Max Sharpe vs Min Variance',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
    ax.grid(axis='y', alpha=0.3)
    path = os.path.join(plots_dir, 'portfolio_weights.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path

p2 = plot_portfolio_weights(ms_w, mv_w, clean_tickers)
display(Image(filename=p2))

In [ ]:
def plot_correlation_heatmap(returns, plots_dir=PLOTS_DIR):
    corr = returns.corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, vmin=-1, vmax=1,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_title('Asset Return Correlation Matrix', fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(plots_dir, 'correlation_heatmap.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path

p3 = plot_correlation_heatmap(returns)
display(Image(filename=p3))

---
## 8. Results & Interpretation

In [ ]:
def print_summary(tickers, ms_weights, ms_ret, ms_vol, ms_sharpe,
                   mv_weights, mv_ret, mv_vol, mv_sharpe):
    bar = '─' * 54
    print(f'\n{bar}')
    print(f"  {'Asset':<8}  {'Max Sharpe':>14}  {'Min Variance':>14}")
    print(bar)
    for t, msw, mvw in zip(tickers, ms_weights, mv_weights):
        print(f'  {t:<8}  {msw:>13.1%}  {mvw:>14.1%}')
    print(bar)
    print(f"  {'Return':<8}  {ms_ret:>13.2%}  {mv_ret:>14.2%}")
    print(f"  {'Volatility':<8}  {ms_vol:>13.2%}  {mv_vol:>14.2%}")
    print(f"  {'Sharpe':<8}  {ms_sharpe:>13.2f}  {mv_sharpe:>14.2f}")
    print(bar)
    print('''
┌─ Financial Interpretation ──────────────────────────────────────────────────┐
│                                                                              │
│  MAX SHARPE (Tangency Portfolio)                                             │
│  Maximises return per unit of risk. Concentrates in high-momentum growth    │
│  equities (AAPL, MSFT, AMZN, GOOGL). Ideal for growth-oriented investors   │
│  with a 5+ year horizon who accept higher volatility for higher returns.    │
│                                                                              │
│  MIN VARIANCE (Global Minimum Variance)                                     │
│  Ignores expected returns; minimises total volatility through diversifi-    │
│  cation. Tilts towards defensive assets (JNJ, GLD) and lower-beta names.   │
│  Ideal for conservative investors prioritising capital preservation.        │
│                                                                              │
│  THE EFFICIENT FRONTIER                                                      │
│  Every portfolio on the frontier is undominated. Portfolios below it are   │
│  sub-optimal; those above are unattainable given this asset universe.       │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘''')

print_summary(clean_tickers,
              ms_w, ms_ret, ms_vol, ms_sharpe,
              mv_w, mv_ret, mv_vol, mv_sharpe)

---
## Conclusion & Next Steps

This notebook demonstrates the complete Markowitz pipeline end-to-end.

### What we proved
- Diversification genuinely reduces risk: the efficient frontier lies **above and to the left** of any individual asset
- Two objective functions (Max Sharpe vs Min Variance) lead to structurally different allocations, each appropriate for a different investor risk profile
- The scipy SLSQP solver reliably converges to global optima for this convex problem

### Limitations & Extensions
| Limitation | Possible Extension |
|------------|-------------------|
| Historical returns ≠ future returns | Black-Litterman model to blend views with market priors |
| Normal return distribution assumed | Fat-tail modelling with CVaR or Expected Shortfall |
| Long-only, no leverage | Allow short positions; add leverage constraints |
| Static covariance matrix | Rolling or DCC-GARCH dynamic covariance |
| No transaction costs | Add turnover penalty to the objective function |
| Single-period model | Multi-period dynamic rebalancing optimisation |